In [2]:
import pandas as pd

In [7]:

df = pd.read_csv("../data/student-mat.csv")


In [8]:
df.head

<bound method NDFrame.head of     school sex  age address famsize Pstatus  Medu  Fedu      Mjob      Fjob  \
0       GP   F   18       U     GT3       A     4     4   at_home   teacher   
1       GP   F   17       U     GT3       T     1     1   at_home     other   
2       GP   F   15       U     LE3       T     1     1   at_home     other   
3       GP   F   15       U     GT3       T     4     2    health  services   
4       GP   F   16       U     GT3       T     3     3     other     other   
..     ...  ..  ...     ...     ...     ...   ...   ...       ...       ...   
390     MS   M   20       U     LE3       A     2     2  services  services   
391     MS   M   17       U     LE3       T     3     1  services  services   
392     MS   M   21       R     GT3       T     1     1     other     other   
393     MS   M   18       R     LE3       T     3     2  services     other   
394     MS   M   19       U     LE3       T     1     1     other   at_home   

     ... famrel freet

In [9]:
cols = ["traveltime", "studytime", "absences", "failures", "Dalc", "health", "G3"]
bn_df = df[cols].copy()

bn_df.head()

,traveltime,studytime,absences,failures,Dalc,health,G3
0,2,2,6,0,1,3,6
1,1,2,4,0,1,3,6
2,1,2,10,3,2,3,10
3,1,3,2,0,1,5,15
4,1,2,4,0,1,5,10


In [10]:
print(bn_df["absences"].describe())
print(bn_df["G3"].describe())

print(bn_df["absences"].value_counts().sort_index())
print(bn_df["G3"].value_counts().sort_index())


count    395.000000
mean       5.708861
std        8.003096
min        0.000000
25%        0.000000
50%        4.000000
75%        8.000000
max       75.000000
Name: absences, dtype: float64
count    395.000000
mean      10.415190
std        4.581443
min        0.000000
25%        8.000000
50%       11.000000
75%       14.000000
max       20.000000
Name: G3, dtype: float64
absences
0     115
1       3
2      65
3       8
4      53
5       5
6      31
7       7
8      22
9       3
10     17
11      3
12     12
13      3
14     12
15      3
16      7
17      1
18      5
19      1
20      4
21      1
22      3
23      1
24      1
25      1
26      1
28      1
30      1
38      1
40      1
54      1
56      1
75      1
Name: count, dtype: int64
G3
0     38
4      1
5      7
6     15
7      9
8     32
9     28
10    56
11    47
12    31
13    31
14    27
15    33
16    16
17     6
18    12
19     5
20     1
Name: count, dtype: int64


In [11]:
# absences 分箱
def bin_absences(x):
    if x <= 2:
        return "low"
    elif x <= 6:
        return "medium"
    else:
        return "high"

# G3 分箱
def bin_g3(x):
    if x <= 9:
        return "low"
    elif x <= 13:
        return "medium"
    else:
        return "high"

bn_df["absences_cat"] = bn_df["absences"].apply(bin_absences)
bn_df["G3_cat"] = bn_df["G3"].apply(bin_g3)

In [12]:
bn_model_df = bn_df[
    ["traveltime", "studytime", "absences_cat", "failures", "Dalc", "health", "G3_cat"]
].copy()

bn_model_df.head()

,traveltime,studytime,absences_cat,failures,Dalc,health,G3_cat
0,2,2,medium,0,1,3,low
1,1,2,medium,0,1,3,low
2,1,2,high,3,2,3,medium
3,1,3,low,0,1,5,high
4,1,2,medium,0,1,5,medium


In [13]:
for col in bn_model_df.columns:
    bn_model_df[col] = bn_model_df[col].astype(str)

In [17]:
from pgmpy.models import DiscreteBayesianNetwork


In [15]:

edges = [
    ("traveltime", "studytime"),
    ("traveltime", "absences_cat"),
    ("studytime", "G3_cat"),
    ("absences_cat", "G3_cat"),
    ("failures", "G3_cat"),
    ("Dalc", "health"),
    ("health", "G3_cat"),
]

In [18]:
model = DiscreteBayesianNetwork(edges)

In [19]:
from pgmpy.estimators import MaximumLikelihoodEstimator

e:\Bayesian\02_Bayedian Network\.venv\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [20]:
model.fit(bn_model_df, estimator=MaximumLikelihoodEstimator)

In [21]:
model.check_model()

True

In [22]:
cpd_G3 = model.get_cpds("G3_cat")
print(cpd_G3)

+----------------+-----+----------------------+
| absences_cat   | ... | absences_cat(medium) |
+----------------+-----+----------------------+
| failures       | ... | failures(3)          |
+----------------+-----+----------------------+
| health         | ... | health(5)            |
+----------------+-----+----------------------+
| studytime      | ... | studytime(4)         |
+----------------+-----+----------------------+
| G3_cat(high)   | ... | 0.3333333333333333   |
+----------------+-----+----------------------+
| G3_cat(low)    | ... | 0.3333333333333333   |
+----------------+-----+----------------------+
| G3_cat(medium) | ... | 0.3333333333333333   |
+----------------+-----+----------------------+


In [23]:
from pgmpy.inference import VariableElimination

infer = VariableElimination(model)

In [24]:
q1 = infer.query(
    variables=["G3_cat"],
    evidence={"studytime": "4"}
)

print(q1)

+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.3419 |
+----------------+---------------+
| G3_cat(low)    |        0.3444 |
+----------------+---------------+
| G3_cat(medium) |        0.3136 |
+----------------+---------------+


In [25]:
q2 = infer.query(
    variables=["G3_cat"],
    evidence={"studytime": "1"}
)

print(q2)

+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.2368 |
+----------------+---------------+
| G3_cat(low)    |        0.2729 |
+----------------+---------------+
| G3_cat(medium) |        0.4903 |
+----------------+---------------+


In [26]:
infer.query(
    variables=["G3_cat"],
    evidence={"absences_cat": "high"}
)

<DiscreteFactor representing phi(G3_cat:3) at 0x25dc4923b50>

In [27]:
infer.query(
    variables=["G3_cat"],
    evidence={"Dalc": "5"}
)

<DiscreteFactor representing phi(G3_cat:3) at 0x25dc49216d0>

In [28]:
infer.query(
    variables=["G3_cat"],
    evidence={
        "studytime": "4",
        "failures": "0",
        "absences_cat": "low"
    }
)

<DiscreteFactor representing phi(G3_cat:3) at 0x25dc496ca90>

In [29]:
q_abs = infer.query(
    variables=["G3_cat"],
    evidence={"absences_cat": "high"}
)
print(q_abs)

q_dalc = infer.query(
    variables=["G3_cat"],
    evidence={"Dalc": "5"}
)
print(q_dalc)

q_combo = infer.query(
    variables=["G3_cat"],
    evidence={
        "studytime": "4",
        "failures": "0",
        "absences_cat": "low"
    }
)
print(q_combo)

+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.2073 |
+----------------+---------------+
| G3_cat(low)    |        0.3333 |
+----------------+---------------+
| G3_cat(medium) |        0.4594 |
+----------------+---------------+
+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.2554 |
+----------------+---------------+
| G3_cat(low)    |        0.2993 |
+----------------+---------------+
| G3_cat(medium) |        0.4453 |
+----------------+---------------+
+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.3322 |
+----------------+---------------+
| G3_cat(low)    |        0.4900 |
+----------------+---------------+
| G3_cat(medium) |        0.1779 |
+----------------+---------------+


In [30]:
def show_query_result(infer, variable, evidence):
    q = infer.query(variables=[variable], evidence=evidence)
    print(f"Query: P({variable} | {evidence})")
    print(q)
    print("-" * 50)
    return q

In [32]:
show_query_result(infer, "G3_cat", {"absences_cat": "high"})
show_query_result(infer, "G3_cat", {"absences_cat": "low"})
show_query_result(infer, "G3_cat", {"Dalc": "5"})
show_query_result(infer, "G3_cat", {"Dalc": "1"})
show_query_result(infer, "G3_cat", {"failures": "0"})
show_query_result(infer, "G3_cat", {"failures": "3"})
show_query_result(infer, "G3_cat", {"studytime": "1"})
show_query_result(infer, "G3_cat", {"studytime": "4"})


show_query_result(
    infer,
    "G3_cat",
    {"studytime": "4", "failures": "0", "absences_cat": "low"}
)

Query: P(G3_cat | {'absences_cat': 'high'})
+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.2073 |
+----------------+---------------+
| G3_cat(low)    |        0.3333 |
+----------------+---------------+
| G3_cat(medium) |        0.4594 |
+----------------+---------------+
--------------------------------------------------
Query: P(G3_cat | {'absences_cat': 'low'})
+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.2959 |
+----------------+---------------+
| G3_cat(low)    |        0.3152 |
+----------------+---------------+
| G3_cat(medium) |        0.3890 |
+----------------+---------------+
--------------------------------------------------
Query: P(G3_cat | {'Dalc': '5'})
+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.2554 |
+-------

<DiscreteFactor representing phi(G3_cat:3) at 0x25dc4842250>

Inference Results

The Bayesian Network reveals that studytime has a strong positive association with final grades. The probability of achieving a high grade increases from 23.7% to 34.2% when studytime changes from level 1 to level 4.

Absences also show a clear negative effect. Students with low absences have a significantly higher probability of high grades (29.6%) compared to those with high absences (20.7%).

Past failures exhibit a strong negative relationship with performance. Students with multiple past failures are much more likely to fall into the low-performance category.

Interestingly, daily alcohol consumption (Dalc) shows minimal effect on final grades in this model. This is likely because its influence is modeled indirectly through health, which weakens its observable impact.

Even under favorable conditions (high studytime, no failures, low absences), the model still assigns a relatively high probability to low performance. This suggests that important variables influencing academic performance are not included in the model.

#### P(G3_cat∣do(studytime=4))

In [33]:
from copy import deepcopy
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

In [34]:
def intervene_on_variable(model, variable, fixed_state, state_names_map):
    """
    對 Bayesian Network 做 intervention: do(variable = fixed_state)

    Parameters
    ----------
    model : pgmpy.models.BayesianNetwork
        原始已 fit 好的模型
    variable : str
        要介入的變數名稱
    fixed_state : str
        要固定成的狀態，例如 "4"
    state_names_map : dict
        每個節點的 state 順序，例如:
        {
            "studytime": ["1", "2", "3", "4"],
            ...
        }

    Returns
    -------
    intervened_model : BayesianNetwork
        介入後的新模型
    """
    intervened_model = deepcopy(model)

    # 1. 移除所有進入 variable 的邊
    parents = list(intervened_model.get_parents(variable))
    for p in parents:
        intervened_model.remove_edge(p, variable)

    # 2. 移除原本 variable 的 CPD
    intervened_model.remove_cpds(variable)

    # 3. 建立新的 deterministic CPD
    states = state_names_map[variable]
    probs = [[1.0 if s == fixed_state else 0.0] for s in states]

    new_cpd = TabularCPD(
        variable=variable,
        variable_card=len(states),
        values=probs,
        state_names={variable: states}
    )

    intervened_model.add_cpds(new_cpd)

    return intervened_model

In [35]:
state_names_map = {
    "traveltime": sorted(bn_model_df["traveltime"].unique()),
    "studytime": sorted(bn_model_df["studytime"].unique()),
    "absences_cat": sorted(bn_model_df["absences_cat"].unique()),
    "failures": sorted(bn_model_df["failures"].unique()),
    "Dalc": sorted(bn_model_df["Dalc"].unique()),
    "health": sorted(bn_model_df["health"].unique()),
    "G3_cat": sorted(bn_model_df["G3_cat"].unique()),
}

state_names_map

{'traveltime': ['1', '2', '3', '4'],
 'studytime': ['1', '2', '3', '4'],
 'absences_cat': ['high', 'low', 'medium'],
 'failures': ['0', '1', '2', '3'],
 'Dalc': ['1', '2', '3', '4', '5'],
 'health': ['1', '2', '3', '4', '5'],
 'G3_cat': ['high', 'low', 'medium']}

In [36]:
do_model_study_4 = intervene_on_variable(
    model=model,
    variable="studytime",
    fixed_state="4",
    state_names_map=state_names_map
)

do_model_study_4.check_model()

True

In [37]:
infer_do_study_4 = VariableElimination(do_model_study_4)

In [38]:
q_do_study_4 = infer_do_study_4.query(variables=["G3_cat"])
print(q_do_study_4)

+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.3422 |
+----------------+---------------+
| G3_cat(low)    |        0.3433 |
+----------------+---------------+
| G3_cat(medium) |        0.3145 |
+----------------+---------------+


In [39]:
do_model_study_1 = intervene_on_variable(
    model=model,
    variable="studytime",
    fixed_state="1",
    state_names_map=state_names_map
)

infer_do_study_1 = VariableElimination(do_model_study_1)

q_do_study_1 = infer_do_study_1.query(variables=["G3_cat"])
print(q_do_study_1)

+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.2359 |
+----------------+---------------+
| G3_cat(low)    |        0.2734 |
+----------------+---------------+
| G3_cat(medium) |        0.4908 |
+----------------+---------------+


In [42]:
q_obs_study_4 = infer.query(
    variables=["G3_cat"],
    evidence={"studytime": "4"}
)

q_obs_study_1 = infer.query(
    variables=["G3_cat"],
    evidence={"studytime": "1"}
)

print(q_obs_study_1)
print(q_obs_study_4)

+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.2368 |
+----------------+---------------+
| G3_cat(low)    |        0.2729 |
+----------------+---------------+
| G3_cat(medium) |        0.4903 |
+----------------+---------------+
+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.3419 |
+----------------+---------------+
| G3_cat(low)    |        0.3444 |
+----------------+---------------+
| G3_cat(medium) |        0.3136 |
+----------------+---------------+


In [43]:
do_model_abs_low = intervene_on_variable(
    model=model,
    variable="absences_cat",
    fixed_state="low",
    state_names_map=state_names_map
)

infer_do_abs_low = VariableElimination(do_model_abs_low)
q_do_abs_low = infer_do_abs_low.query(variables=["G3_cat"])
print(q_do_abs_low)

+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.2959 |
+----------------+---------------+
| G3_cat(low)    |        0.3151 |
+----------------+---------------+
| G3_cat(medium) |        0.3890 |
+----------------+---------------+


In [44]:
do_model_abs_high = intervene_on_variable(
    model=model,
    variable="absences_cat",
    fixed_state="high",
    state_names_map=state_names_map
)

infer_do_abs_high = VariableElimination(do_model_abs_high)
q_do_abs_high = infer_do_abs_high.query(variables=["G3_cat"])
print(q_do_abs_high)

+----------------+---------------+
| G3_cat         |   phi(G3_cat) |
+================+===============+
| G3_cat(high)   |        0.2065 |
+----------------+---------------+
| G3_cat(low)    |        0.3336 |
+----------------+---------------+
| G3_cat(medium) |        0.4599 |
+----------------+---------------+


Causal Interpretation

The interventional distributions P(G3_cat∣do(studytime)) and P(G3_cat∣do(absences_cat))

P(G3_cat∣do(absences_cat)) are very close to their observational counterparts.

This suggests that, under the current Bayesian Network structure, the observed associations for studytime and absences are already close to their estimated causal effects.

A likely explanation is that the manually specified DAG contains relatively limited confounding structure, so conditioning and intervention lead to similar results.

This does not mean that intervention is unnecessary. Rather, it shows that in this simplified graph, the variables of interest are not heavily confounded by upstream factors.

Limitations

The causal conclusions depend strongly on the correctness of the manually specified DAG. Since the graph is simplified and omits many potential confounders, the estimated intervention effects should be interpreted as causal effects under the assumed model, rather than definitive real-world causal truth.